In [ ]:
# --- ensure repo-root cwd ---
import os, sys
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src'))
import warnings; warnings.filterwarnings('ignore')  # silence sklearn feature-name noise

# GPU pipeline — cuDF coulomb counting + GPyTorch GPR + Chronos forecasting

Uses the RTX 5090 for the three places GPU actually helps:
1. **cuDF** — coulomb counting over the 13M-row intellicar extract on GPU.
2. **GPyTorch** — exact GP regression on GPU for SoH estimation *with uncertainty*.
3. **Chronos** (Amazon foundation model) — per-vehicle SoH trajectory forecasting → RUL with
   prediction intervals.

## 0. GPU data pipeline — coulomb counting on cuDF vs CPU

In [ ]:
import numpy as np, pandas as pd, time, torch
import pyarrow.dataset as ds
import soh
print("GPU:", torch.cuda.get_device_name(0))

ic = ds.dataset("data/mahindra/intellicar", format="parquet").to_table(
        columns=["vin", "eventAt", "soc", "current"]).to_pandas()
ic["t"] = pd.to_datetime(pd.to_numeric(ic["eventAt"], errors="coerce"), unit="ms")
for c in ["soc", "current"]:
    ic[c] = pd.to_numeric(ic[c], errors="coerce")
ic = ic.dropna(subset=["vin", "t", "soc", "current"]).query("0 <= soc <= 100")[["vin", "t", "soc", "current"]]
print(f"{len(ic):,} rows")

t0 = time.time(); cpu_cap, _ = soh.coulomb_capacity_monthly(ic, backend="cpu"); t_cpu = time.time() - t0
t0 = time.time(); gpu_cap, used = soh.coulomb_capacity_monthly(ic, backend="gpu"); t_gpu = time.time() - t0
print(f"CPU vectorized: {t_cpu:.1f}s | cuDF GPU: {t_gpu:.1f}s (used_gpu={used})")
print(f"results match: {np.allclose(cpu_cap['capacity_ah'].sort_values().values, gpu_cap['capacity_ah'].sort_values().values, atol=1)}")
target = soh.capacity_to_soh(gpu_cap if used else cpu_cap)

## 1. GPyTorch exact GP on GPU — SoH estimation with uncertainty

ARD-RBF kernel, leakage-safe grouped CV by VIN. Reports MAE and the mean predictive σ
(the uncertainty that feeds RUL confidence bounds).

In [ ]:
import gpytorch
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
dev = torch.device("cuda")

m = pd.read_parquet("data/mahindra/features/feature_table.parquet")
FEATURES = ["age_months", "cum_ah", "cum_km", "odo_max", "ah_throughput", "cur_abs_mean",
            "cur_abs_p95", "cur_dis_mean", "cur_chg_mean", "soc_mean", "frac_soc_high",
            "frac_soc_low", "volt_mean", "volt_min", "volt_max", "temp_mean", "temp_max",
            "lat_mean", "lon_mean", "frac_drive", "dte_mean"]
Xdf = m[FEATURES].fillna(m[FEATURES].median()); y = m["soh"].to_numpy()
Xz = ((Xdf - Xdf.mean()) / (Xdf.std() + 1e-9)).to_numpy()

class GP(gpytorch.models.ExactGP):
    def __init__(self, x, y, lik):
        super().__init__(x, y, lik)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=x.shape[1]))
    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(self.mean_module(x), self.covar_module(x))

def fit_predict(Xtr, ytr, Xte):
    xtr = torch.tensor(Xtr, dtype=torch.float32, device=dev)
    ytr_t = torch.tensor(ytr, dtype=torch.float32, device=dev)
    lik = gpytorch.likelihoods.GaussianLikelihood().to(dev)
    model = GP(xtr, ytr_t, lik).to(dev)
    model.train(); lik.train()
    opt = torch.optim.Adam(model.parameters(), lr=0.1)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(lik, model)
    for _ in range(80):
        opt.zero_grad(); out = model(xtr); loss = -mll(out, ytr_t); loss.backward(); opt.step()
    model.eval(); lik.eval()
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred = lik(model(torch.tensor(Xte, dtype=torch.float32, device=dev)))
    return pred.mean.cpu().numpy(), pred.stddev.cpu().numpy()

gkf = GroupKFold(n_splits=min(5, m["vin"].nunique()))
pred = np.full(len(y), np.nan); sig = np.full(len(y), np.nan)
for tr, te in gkf.split(Xz, y, m["vin"]):
    pred[te], sig[te] = fit_predict(Xz[tr], y[tr], Xz[te])
print(f"GPyTorch GPU-GPR  MAE {mean_absolute_error(y, pred):.2f}  R2 {r2_score(y, pred):.3f}  "
      f"mean sigma {np.nanmean(sig):.2f} pp")

## 2. Chronos (foundation model) — forecast SoH trajectory → RUL with intervals

For each vehicle we feed its monthly SoH history to Chronos-Bolt (on GPU) and forecast 24 months
ahead. RUL = months until the **median** forecast (made monotonic) crosses the 80% EOL; the
10–90% quantile band gives the confidence range.

In [ ]:
from chronos import BaseChronosPipeline
pipe = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-base",
                                           device_map="cuda", torch_dtype=torch.bfloat16)
H, EOL = 24, 80.0
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 5))
rows = []
for i, (vin, g) in enumerate(target.sort_values("month").groupby("vin")):
    s = g["soh"].to_numpy()
    if len(s) < 6:
        continue
    q, mean = pipe.predict_quantiles(torch.tensor(s, dtype=torch.float32),
                                     prediction_length=H, quantile_levels=[0.1, 0.5, 0.9])
    lo, med, hi = [np.minimum.accumulate(q[0, :, j].cpu().float().numpy()) for j in range(3)]
    months = np.arange(1, H + 1)
    below = np.where(med <= EOL)[0]
    rul = int(below[0] + 1) if len(below) else None
    rows.append({"vin": vin, "soh_now": round(float(s[-1]), 1),
                 "RUL_mo_med": rul, "soh_in_24mo": round(float(med[-1]), 1)})
    if i < 4:  # plot a few
        base = len(s)
        ax.plot(np.arange(base), s, marker=".", color=f"C{i}", label=vin[-6:])
        ax.plot(np.arange(base, base + H), med, "--", color=f"C{i}")
        ax.fill_between(np.arange(base, base + H), lo, hi, color=f"C{i}", alpha=0.15)
ax.axhline(EOL, ls=":", color="red"); ax.set_xlabel("month index"); ax.set_ylabel("SoH (%)")
ax.set_title("Chronos SoH forecast (dashed = median, band = 10–90%)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

rul_gpu = pd.DataFrame(rows).sort_values("RUL_mo_med", na_position="last")
print(rul_gpu.to_string(index=False))
rul_gpu.to_csv("data/mahindra/soh/rul_chronos.csv", index=False)